In [1]:
# 1. Aplinkos paruošimas ir bibliotekų diegimas
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2 cmake

# Įdiegiame TensorFlow su senojo Keras suderinamumu
!pip install -q tensorflow==2.16.1 tf-keras~=2.16

# Per jėgą atnaujiname ml_dtypes, kad nelūžtų JAX (Google Colab klaidos ištaisymas)
!pip install -q --upgrade ml_dtypes jax jaxlib

# Įdiegiame garso apdorojimo bibliotekas
!pip install -q gtts librosa soundfile huggingface_hub

# Įdiegiame originalią ir pačią stabiliausią microWakeWord versiją
!pip install -q git+https://github.com/kahrendt/microWakeWord.git

print("✅ Bibliotekos sėkmingai įdiegtos!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libportaudio2:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libportaudio2_19.6.0-1.1_amd64.deb ...
Unpacking libportaudio2:amd64 (19.6.0-1.1) ...
Setting up libportaudio2:amd64 (19.6.0-1.1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic 

In [6]:
# 2. Duomenų generavimas ir fono paruošimas
# Dar kartą priverstinai įdiegiame trūkstamas bibliotekas šiam žingsniui
!pip install -q gtts librosa soundfile huggingface_hub

import os
import random
import warnings
import librosa
import soundfile as sf
from gtts import gTTS
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")
target_word = "asista"
output_dir = "/content/positive_samples"
negative_data_dir = "/content/negative_features"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(negative_data_dir, exist_ok=True)

print("1/3: Generuojamas bazinis 'asista' garsas...")
base_audio_path = "base_asista.mp3"
tts = gTTS(text=target_word, lang='lt')
tts.save(base_audio_path)

print("2/3: Kuriamos 2500 balso variacijų (gali užtrukti apie 1-2 min)...")
y, sr = librosa.load(base_audio_path, sr=16000)
for i in range(2500):
    rate = random.uniform(0.85, 1.25)  # keičiamas greitis
    steps = random.uniform(-5, 5)      # keičiamas balso tonas
    y_mod = librosa.effects.time_stretch(y, rate=rate)
    y_mod = librosa.effects.pitch_shift(y_mod, sr=sr, n_steps=steps)
    sf.write(f"{output_dir}/asista_{i}.wav", y_mod, sr)

print("3/3: Atsisiunčiami fono triukšmai...")
try:
    snapshot_download(
        repo_id="OHF-Voice/mww-feature-data",
        repo_type="dataset",
        local_dir=negative_data_dir,
        allow_patterns=["*.mmap", "*.json"],
        max_workers=4
    )
    print("✅ Naudojami OHF-Voice fono garsai!")
except Exception:
    print("⚠️ HuggingFace nepasiekiamas. Atsisiunčiamas Google fono garsų paketas...")
    os.system("wget -q https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz")
    os.system(f"tar -xzf speech_commands_v0.02.tar.gz -C {negative_data_dir} _background_noise_")
    os.system("rm speech_commands_v0.02.tar.gz")

print("✅ Visi duomenys sėkmingai sugeneruoti ir paruošti!")

1/3: Generuojamas bazinis 'asista' garsas...
2/3: Kuriamos 2500 balso variacijų (gali užtrukti apie 1-2 min)...
3/3: Atsisiunčiami fono triukšmai...
⚠️ HuggingFace nepasiekiamas. Atsisiunčiamas Google fono garsų paketas...
✅ Visi duomenys sėkmingai sugeneruoti ir paruošti!


In [3]:
# 3. Paklausome sugeneruoto balso
import IPython.display as ipd
import glob
import random

failai = glob.glob("/content/positive_samples/*.wav")
if failai:
    atsitiktinis = random.choice(failai)
    print(f"Klausomės variacijos: {atsitiktinis.split('/')[-1]}")
    display(ipd.Audio(atsitiktinis))
else:
    print("❌ Klaida: garso failai nesugeneruoti.")

Klausomės variacijos: asista_2093.wav


In [40]:
import os
import sys

# 1. Aplinkos paruošimas
%cd /content
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["PYTHONPATH"] = "/content/microWakeWord:" + os.environ.get("PYTHONPATH", "")

# 2. Duomenų struktūros sutvarkymas (SVARBU!)
# Sukuriame kelius, kurių tikisi FeatureHandler
!mkdir -p /content/positive_samples/asista
!cp /content/positive_samples/*.wav /content/positive_samples/asista/ 2>/dev/null

# 3. Sukuriame YAML su visais parametrais požymiams generuoti
yaml_config = """
target_word: "asista"
clip_duration_ms: 1490
sample_rate: 16000
window_size_ms: 30
window_stride_ms: 20

# Šie nustatymai priverčia programą paruošti audio failus
preprocess: "micro"
feature_type: "generate_features"

# Keliai
train_dir: "/content/model_output"
output_dir: "/content/model_output"
positive_data_dir: "/content/positive_samples"
negative_data_dir: "/content/negative_features"
validation_positive_data_dir: "/content/positive_samples"
validation_negative_data_dir: "/content/negative_features"
test_positive_data_dir: "/content/positive_samples"
test_negative_data_dir: "/content/negative_features"

# Treniravimo nustatymai
epochs: 30
batch_size: 128
learning_rate: "0.001,0.0001"
learning_rate_steps: "5000,10000"
"""

with open("/content/training_parameters.yaml", "w") as f:
    f.write(yaml_config)

print("🛠️ 1 ŽINGSNIS: Ruošiami audio požymiai (Features)...")

# 4. Paleidžiame treniravimą per oficialų skriptą
# Naudojame 'inception', nes jis turi mažiausiai papildomų reikalavimų
%cd /content/microWakeWord
!python -m microwakeword.model_train_eval \
    --training_config='/content/training_parameters.yaml' \
    --train=1 \
    --restore_checkpoint=0 \
    inception

print("\n✅ Jei matai 'Step' arba 'Epoch', vadinasi, paruošimas pavyko ir mokymasis vyksta!")

/content
🛠️ 1 ŽINGSNIS: Ruošiami audio požymiai (Features)...
/content/microWakeWord
2026-03-25 22:33:07.233323: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774477987.257568   42649 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774477987.264603   42649 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774477987.282562   42649 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774477987.282659   42649 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0

In [35]:
import os
from google.colab import files

# Nustatome kelius, kur programa turėjo išsaugoti failus
output_path = "/content/model_output"
target_word = "asista"

print("🔍 Ieškoma paruošto modelio failo...")

# Programa dažniausiai sukuria failą su priesaga _stream_quantized.tflite
# arba tiesiog .tflite. Suraskime juos visus.
found_files = []
for root, dirs, filenames in os.walk(output_path):
    for f in filenames:
        if f.endswith(".tflite"):
            found_files.append(os.path.join(root, f))

if found_files:
    print(f"✅ Rasta modelių: {len(found_files)}")
    for f in found_files:
        print(f"⬇️ Atsisiunčiamas: {f}")
        files.download(f)
else:
    # Jei programa dar nebaigė arba neįrašė .tflite, bandom paimti .keras versiją
    print("⚠️ TFLite failų nerasta. Galbūt treniravimas nebaigtas?")
    keras_file = f"/content/microWakeWord/{target_word}_model.keras"
    if os.path.exists(keras_file):
        print(f"⬇️ Atsisiunčiamas bazinis Keras modelis: {keras_file}")
        files.download(keras_file)
    else:
        print("❌ Klaida: jokių modelio failų nerasta. Patikrinkite 'model_output' aplanką kairėje.")

🔍 Ieškoma paruošto modelio failo...
⚠️ TFLite failų nerasta. Galbūt treniravimas nebaigtas?
❌ Klaida: jokių modelio failų nerasta. Patikrinkite 'model_output' aplanką kairėje.
